# Provenance

- Cohort: Reconstructed document-disjoint audit cohort.
- Purpose: Measure document and query overlap between DORIS-MAE training exposure and LitSearch/CSFCube evaluation material.
- Matching configuration: `configs/experiments/document_disjoint.yaml`.
- External inputs: Externally supplied DORIS-MAE, LitSearch, and CSFCube files under `external_data/`.
- Expected output type: CSV and JSON leakage-audit reports under `results/leakage_analysis/`.

All paths below are expressed through repository-relative variables; provide external inputs locally without committing them.


In [ ]:
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()
EXTERNAL_DATA_DIR = REPOSITORY_ROOT / "external_data"
RESULTS_DIR = REPOSITORY_ROOT / "results"
DORIS_MAE_DIR = EXTERNAL_DATA_DIR / "doris_mae"
CSFCUBE_DATA_DIR = EXTERNAL_DATA_DIR / "csfcube"
LITSEARCH_DATA_DIR = EXTERNAL_DATA_DIR / "litsearch"
LEAKAGE_OUTPUT_DIR = RESULTS_DIR / "leakage_analysis"


# Document-level overlap and leakage analysis

This notebook checks document-level overlap between the DORIS-MAE training material and the LitSearch / CSFCube evaluation material used in this project.

The analysis is intentionally conservative:

- DORIS-MAE training exposure is the union of every document in `HRIA/reranking_dataset_doris_mae.jsonl` candidate pools, not only the sampled 50-document prompt variants.
- CSFCube is checked with local query/candidate pools and local title/abstract text.
- LitSearch is checked from local retrieval/reranking result IDs. If the Hugging Face `princeton-nlp/LitSearch` corpus is locally available, the notebook also performs text-hash matching for LitSearch.
- Raw integer IDs are not treated as sufficient proof across datasets, because DORIS-MAE uses local `abstract_id`s while LitSearch/CSFCube use other paper-ID namespaces. Exact normalized title and title+abstract matches are stronger leakage evidence.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import re
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Iterable

DORIS_DIR = DORIS_MAE_DIR / "splits"
HRIA = DORIS_MAE_DIR
CSFCUBE_DIR = CSFCUBE_DATA_DIR
LITSEARCH_DIR = LITSEARCH_DATA_DIR
OUT_DIR = LEAKAGE_OUTPUT_DIR
OUT_DIR.mkdir(exist_ok=True)

print("Repository root:", REPOSITORY_ROOT)
print("Output dir:", OUT_DIR)


## Helpers

The helpers below normalize text, extract common metadata identifiers, and load JSON/JSONL files. Title hashes and title+abstract hashes are used for exact document matching after normalization.

In [ ]:
def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def iter_jsonl(path: Path) -> Iterable[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def normalize_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        value = " ".join(str(x) for x in value if x is not None)
    value = str(value).lower()
    value = value.replace("_", " ")
    value = re.sub(r"https?://\S+", " ", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def stable_hash(value: Any) -> str:
    normalized = normalize_text(value)
    if not normalized:
        return ""
    return hashlib.sha1(normalized.encode("utf-8")).hexdigest()


def first_nonempty(*values: Any) -> str:
    for value in values:
        if isinstance(value, list):
            value = " ".join(str(x) for x in value if x is not None)
        if value is not None and str(value).strip():
            return str(value).strip()
    return ""


def extract_arxiv_id(*values: Any) -> str:
    text = " ".join(str(v) for v in values if v is not None)
    match = re.search(r"(?:arxiv\.org/(?:abs|pdf)/|arxiv:)?(\d{4}\.\d{4,5})(?:v\d+)?", text, re.I)
    return match.group(1) if match else ""


def normalize_doi(value: Any) -> str:
    if not value:
        return ""
    text = str(value).lower().strip()
    text = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", text)
    text = re.sub(r"^doi:\s*", "", text)
    return text.strip()


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    fieldnames = list(rows[0].keys())
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def write_json(path: Path, obj: Any) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def maybe_display(rows: list[dict[str, Any]], limit: int = 20) -> None:
    try:
        import pandas as pd
        display(pd.DataFrame(rows).head(limit))
    except Exception:
        for row in rows[:limit]:
            print(row)


## Load DORIS-MAE exposure sets

`doris_train_docs` contains all DORIS documents that occur in the generated aspect-level reranking dataset candidate pools. This is the key training-exposure set for leakage analysis.

In [ ]:
doris_corpus = load_json(DORIS_DIR / "Corpus.json")
doris_by_abstract_id: dict[str, dict[str, Any]] = {}

for idx, doc in enumerate(doris_corpus):
    abstract_id = str(doc.get("abstract_id", idx))
    doris_by_abstract_id[abstract_id] = doc

doris_train_candidate_ids: set[str] = set()
doris_positive_candidate_ids: set[str] = set()
doris_rows = 0

for row in iter_jsonl(HRIA / "reranking_dataset_doris_mae.jsonl"):
    doris_rows += 1
    for doc_id, score in zip(row["candidate_pool"], row["candidate_scores"]):
        doc_id = str(doc_id)
        doris_train_candidate_ids.add(doc_id)
        if int(score) > 0:
            doris_positive_candidate_ids.add(doc_id)

def make_doris_record(abstract_id: str, role: str) -> dict[str, Any]:
    doc = doris_by_abstract_id[abstract_id]
    title = first_nonempty(doc.get("title"))
    abstract = first_nonempty(doc.get("original_abstract"), doc.get("masked_abstract"))
    url = first_nonempty(doc.get("url"))
    return {
        "dataset": "DORIS-MAE",
        "role": role,
        "doc_id": abstract_id,
        "source_id": first_nonempty(doc.get("ss_id")),
        "arxiv_id": extract_arxiv_id(url, title),
        "doi": normalize_doi(doc.get("doi")),
        "title": title,
        "abstract": abstract,
        "title_hash": stable_hash(title),
        "text_hash": stable_hash(title + " " + abstract),
    }

doris_train_docs = [make_doris_record(doc_id, "train_candidate") for doc_id in sorted(doris_train_candidate_ids, key=int)]
doris_positive_docs = [make_doris_record(doc_id, "positive_train_candidate") for doc_id in sorted(doris_positive_candidate_ids, key=int)]
doris_corpus_docs = [make_doris_record(doc_id, "corpus") for doc_id in sorted(doris_by_abstract_id, key=int)]

print("DORIS aspect-level rows:", doris_rows)
print("DORIS corpus docs:", len(doris_corpus_docs))
print("DORIS train-exposed unique candidate docs:", len(doris_train_docs))
print("DORIS positive train-exposed unique docs:", len(doris_positive_docs))


## Load CSFCube evaluation documents

CSFCube has local title/abstract text, so we can check ID metadata, exact normalized title matches, exact title+abstract matches, and high-similarity title matches.

In [ ]:
csf_text_by_pid: dict[str, dict[str, Any]] = {}
for doc in iter_jsonl(CSFCUBE_DIR / "abstracts-csfcube-preds.jsonl"):
    pid = str(doc.get("paper_id"))
    csf_text_by_pid[pid] = doc

csf_roles: dict[str, set[str]] = defaultdict(set)
csf_facets: dict[str, set[str]] = defaultdict(set)

for facet in ["background", "method", "result"]:
    data = load_json(CSFCUBE_DIR / f"test-pid2anns-csfcube-{facet}.json")
    for query_pid, payload in data.items():
        query_pid = str(query_pid)
        csf_roles[query_pid].add("query")
        csf_facets[query_pid].add(facet)
        for cand_pid in payload.get("cands", []):
            cand_pid = str(cand_pid)
            csf_roles[cand_pid].add("candidate")
            csf_facets[cand_pid].add(facet)

def make_csf_record(pid: str) -> dict[str, Any]:
    doc = csf_text_by_pid.get(pid, {})
    metadata = doc.get("metadata", {}) if isinstance(doc.get("metadata"), dict) else {}
    title = first_nonempty(doc.get("title"))
    abstract = first_nonempty(doc.get("abstract"))
    return {
        "dataset": "CSFCube",
        "role": "+".join(sorted(csf_roles.get(pid, {"text_only"}))),
        "doc_id": pid,
        "source_id": pid,
        "arxiv_id": extract_arxiv_id(metadata.get("arxiv_id"), title),
        "doi": normalize_doi(metadata.get("doi")),
        "facets": "+".join(sorted(csf_facets.get(pid, []))),
        "title": title,
        "abstract": abstract,
        "title_hash": stable_hash(title),
        "text_hash": stable_hash(title + " " + abstract),
    }

csf_eval_ids = set(csf_roles)
csf_query_ids = {pid for pid, roles in csf_roles.items() if "query" in roles}
csf_candidate_ids = {pid for pid, roles in csf_roles.items() if "candidate" in roles}
csf_eval_docs = [make_csf_record(pid) for pid in sorted(csf_eval_ids, key=int)]
csf_text_docs = [make_csf_record(pid) for pid in sorted(csf_text_by_pid, key=int)]

print("CSFCube text docs:", len(csf_text_docs))
print("CSFCube eval docs:", len(csf_eval_docs))
print("CSFCube query docs:", len(csf_query_ids))
print("CSFCube candidate docs:", len(csf_candidate_ids))


## Load LitSearch local IDs, and optional LitSearch text

Local LitSearch retrieval/reranking files contain query metadata, relevant IDs (`corpusids`), retrieved IDs, and sometimes `pre_reranked`. They usually do not contain document titles/abstracts. If `datasets.load_dataset` can access the LitSearch corpus from cache or network, text matching is enabled automatically.

In [ ]:
lit_roles: dict[str, set[str]] = defaultdict(set)
lit_query_counts = Counter()

lit_result_files = list((LITSEARCH_DIR / "results").rglob("*.jsonl"))
for path in lit_result_files:
    if "metrics" in path.name:
        continue
    for row in iter_jsonl(path):
        lit_query_counts[path.name] += 1
        for corpus_id in row.get("corpusids", []) or []:
            lit_roles[str(corpus_id)].add("gold_relevant")
        for corpus_id in row.get("retrieved", []) or []:
            lit_roles[str(corpus_id)].add("retrieved_or_reranked")
        for corpus_id in row.get("pre_reranked", []) or []:
            lit_roles[str(corpus_id)].add("pre_reranked")

lit_ids = set(lit_roles)
lit_gold_ids = {pid for pid, roles in lit_roles.items() if "gold_relevant" in roles}
lit_retrieved_ids = lit_ids - lit_gold_ids

print("LitSearch local result files:", len(lit_result_files))
print("LitSearch unique local IDs:", len(lit_ids))
print("LitSearch gold relevant IDs:", len(lit_gold_ids))
print("LitSearch retrieved/reranked-only IDs:", len(lit_retrieved_ids))

lit_text_by_pid: dict[str, dict[str, Any]] = {}
lit_text_source = "not_loaded"

try:
    from datasets import load_dataset
    corpus = load_dataset("princeton-nlp/LitSearch", "corpus_clean", split="full")
    for doc in corpus:
        possible_id = first_nonempty(doc.get("corpusid"), doc.get("corpus_id"), doc.get("paper_id"), doc.get("id"))
        if possible_id:
            lit_text_by_pid[str(possible_id)] = dict(doc)
    lit_text_source = "princeton-nlp/LitSearch corpus_clean"
except Exception as exc:
    lit_text_source = f"not_loaded: {type(exc).__name__}: {exc}"

def make_lit_record(pid: str) -> dict[str, Any]:
    doc = lit_text_by_pid.get(pid, {})
    title = first_nonempty(doc.get("title"), doc.get("paper_title"))
    abstract = first_nonempty(doc.get("abstract"), doc.get("text"), doc.get("title_abstract"))
    return {
        "dataset": "LitSearch",
        "role": "+".join(sorted(lit_roles.get(pid, {"text_only"}))),
        "doc_id": pid,
        "source_id": pid,
        # Use explicit metadata only for arXiv IDs. Extracting from the abstract can
        # accidentally match cited papers instead of the current LitSearch document.
        "arxiv_id": extract_arxiv_id(doc.get("arxiv_id"), doc.get("paper_id"), doc.get("url"), title),
        "doi": normalize_doi(doc.get("doi")),
        "title": title,
        "abstract": abstract,
        "title_hash": stable_hash(title),
        "text_hash": stable_hash(title + " " + abstract),
    }

lit_docs = [make_lit_record(pid) for pid in sorted(lit_ids, key=lambda x: int(x) if x.isdigit() else x)]
lit_text_docs = [make_lit_record(pid) for pid in sorted(lit_text_by_pid, key=lambda x: int(x) if x.isdigit() else x)]

print("LitSearch text source:", lit_text_source)
print("LitSearch text docs loaded:", len(lit_text_docs))


## Exact overlap functions

These functions compare two document collections by ID-like fields and by normalized text hashes. Empty identifiers/hashes are ignored.

In [ ]:
def index_by_field(records: list[dict[str, Any]], field: str) -> dict[str, list[dict[str, Any]]]:
    index: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        value = str(record.get(field, "")).strip()
        if value:
            index[value].append(record)
    return index


def overlap_by_field(
    left_name: str,
    left_records: list[dict[str, Any]],
    right_name: str,
    right_records: list[dict[str, Any]],
    field: str,
) -> list[dict[str, Any]]:
    left_index = index_by_field(left_records, field)
    right_index = index_by_field(right_records, field)
    rows = []
    for value in sorted(set(left_index) & set(right_index)):
        for left in left_index[value]:
            for right in right_index[value]:
                rows.append({
                    "match_type": field,
                    "match_value": value,
                    "left_set": left_name,
                    "left_doc_id": left.get("doc_id", ""),
                    "left_role": left.get("role", ""),
                    "left_title": left.get("title", "")[:220],
                    "right_set": right_name,
                    "right_doc_id": right.get("doc_id", ""),
                    "right_role": right.get("role", ""),
                    "right_title": right.get("title", "")[:220],
                })
    return rows


def unique_match_counts(rows: list[dict[str, Any]]) -> tuple[int, int]:
    return len({row["left_doc_id"] for row in rows}), len({row["right_doc_id"] for row in rows})


def run_exact_comparison(
    left_name: str,
    left_records: list[dict[str, Any]],
    right_name: str,
    right_records: list[dict[str, Any]],
) -> dict[str, Any]:
    fields = ["source_id", "arxiv_id", "doi", "title_hash", "text_hash"]
    by_field = {field: overlap_by_field(left_name, left_records, right_name, right_records, field) for field in fields}
    source_left, source_right = unique_match_counts(by_field["source_id"])
    arxiv_left, arxiv_right = unique_match_counts(by_field["arxiv_id"])
    doi_left, doi_right = unique_match_counts(by_field["doi"])
    title_left, title_right = unique_match_counts(by_field["title_hash"])
    text_left, text_right = unique_match_counts(by_field["text_hash"])

    return {
        "left_set": left_name,
        "right_set": right_name,
        "left_docs": len(left_records),
        "right_docs": len(right_records),
        "source_id_pairs": len(by_field["source_id"]),
        "source_id_left_docs": source_left,
        "source_id_right_docs": source_right,
        "arxiv_id_pairs": len(by_field["arxiv_id"]),
        "arxiv_id_left_docs": arxiv_left,
        "arxiv_id_right_docs": arxiv_right,
        "doi_pairs": len(by_field["doi"]),
        "doi_left_docs": doi_left,
        "doi_right_docs": doi_right,
        "title_hash_pairs": len(by_field["title_hash"]),
        "title_hash_left_docs": title_left,
        "title_hash_right_docs": title_right,
        "text_hash_pairs": len(by_field["text_hash"]),
        "text_hash_left_docs": text_left,
        "text_hash_right_docs": text_right,
        "matches": by_field,
    }


comparisons = [
    run_exact_comparison("DORIS train candidates", doris_train_docs, "CSFCube eval docs", csf_eval_docs),
    run_exact_comparison("DORIS positive train candidates", doris_positive_docs, "CSFCube eval docs", csf_eval_docs),
    run_exact_comparison("DORIS train candidates", doris_train_docs, "CSFCube query docs", [r for r in csf_eval_docs if "query" in r["role"]]),
    run_exact_comparison("DORIS train candidates", doris_train_docs, "CSFCube candidate docs", [r for r in csf_eval_docs if "candidate" in r["role"]]),
    run_exact_comparison("DORIS train candidates", doris_train_docs, "LitSearch local IDs", lit_docs),
    run_exact_comparison("DORIS positive train candidates", doris_positive_docs, "LitSearch local IDs", lit_docs),
]

if lit_text_docs:
    comparisons.extend([
        run_exact_comparison("DORIS train candidates", doris_train_docs, "LitSearch text docs", lit_text_docs),
        run_exact_comparison("DORIS positive train candidates", doris_positive_docs, "LitSearch text docs", lit_text_docs),
    ])

summary_rows = [
    {k: v for k, v in comparison.items() if k != "matches"}
    for comparison in comparisons
]

maybe_display(summary_rows)
write_csv(OUT_DIR / "exact_overlap_summary.csv", summary_rows)
write_json(OUT_DIR / "exact_overlap_summary.json", summary_rows)


## Save detailed exact-match files

The summary counts above are usually enough for the thesis. The detailed files below list the matching document IDs/titles for manual inspection.

In [ ]:
detail_rows = []
for comparison in comparisons:
    for field, rows in comparison["matches"].items():
        for row in rows:
            detail_rows.append(row)

write_csv(OUT_DIR / "exact_overlap_details.csv", detail_rows)
write_json(OUT_DIR / "exact_overlap_details.json", detail_rows)

print("Detailed exact matches:", len(detail_rows))
maybe_display(detail_rows, limit=30)


## Near-duplicate title check

This is a lightweight candidate check, not a proof of leakage. It compares normalized titles with blocking to avoid a full quadratic scan. Inspect any high-scoring rows manually.

In [ ]:
def title_block_key(title: str) -> str:
    tokens = [t for t in normalize_text(title).split() if len(t) > 2]
    return " ".join(tokens[:3])


def title_similarity(a: str, b: str) -> float:
    a_norm = normalize_text(a)
    b_norm = normalize_text(b)
    if not a_norm or not b_norm:
        return 0.0
    return SequenceMatcher(None, a_norm, b_norm).ratio()


def near_duplicate_titles(
    left_name: str,
    left_records: list[dict[str, Any]],
    right_name: str,
    right_records: list[dict[str, Any]],
    threshold: float = 0.95,
) -> list[dict[str, Any]]:
    right_blocks: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for right in right_records:
        key = title_block_key(right.get("title", ""))
        if key:
            right_blocks[key].append(right)

    rows = []
    for left in left_records:
        key = title_block_key(left.get("title", ""))
        if not key:
            continue
        for right in right_blocks.get(key, []):
            score = title_similarity(left.get("title", ""), right.get("title", ""))
            if score >= threshold:
                rows.append({
                    "match_type": "near_title",
                    "score": round(score, 4),
                    "left_set": left_name,
                    "left_doc_id": left.get("doc_id", ""),
                    "left_role": left.get("role", ""),
                    "left_title": left.get("title", "")[:220],
                    "right_set": right_name,
                    "right_doc_id": right.get("doc_id", ""),
                    "right_role": right.get("role", ""),
                    "right_title": right.get("title", "")[:220],
                })
    return rows


near_rows = near_duplicate_titles("DORIS train candidates", doris_train_docs, "CSFCube eval docs", csf_eval_docs)
if lit_text_docs:
    near_rows.extend(near_duplicate_titles("DORIS train candidates", doris_train_docs, "LitSearch text docs", lit_text_docs))

near_rows = sorted(near_rows, key=lambda row: row["score"], reverse=True)
write_csv(OUT_DIR / "near_duplicate_title_candidates.csv", near_rows)
write_json(OUT_DIR / "near_duplicate_title_candidates.json", near_rows)

print("Near-duplicate title candidates:", len(near_rows))
maybe_display(near_rows, limit=30)


## Actual prompt-level exposure analysis

The previous document-level analysis is an upper bound: it treats every document in every DORIS-MAE candidate pool as potentially training-exposed. The actual training notebook samples fixed-size lists of 50 candidates, up to 3 variants per aspect-level row.

This section reproduces the deterministic sampling logic from the Gemma training notebooks (`N_VARIANTS=3`, `TARGET_SIZE=50`, `MAX_POSITIVES=5`, `HARD_NEG_TOP=100`, MD5 seed from `row_id || query_text`) and checks overlap only for documents that would actually appear in prompt-formatted training examples.

In [ ]:
# -----------------------------
# Reproduce training-notebook fixed-size augmentation
# -----------------------------

N_VARIANTS = 3
TARGET_SIZE = 50
MAX_POSITIVES = 5
HARD_NEG_TOP = 100
MAX_TOTAL_TRIES = 3000


def training_stable_seed(row_id: int, query_text: str) -> int:
    h = hashlib.md5(f"{row_id}||{query_text}".encode("utf-8")).hexdigest()
    return int(h[:8], 16)


def training_pool_signature(doc_ids: list[Any]) -> tuple[int, ...]:
    return tuple(sorted(int(d) for d in doc_ids))


def sample_training_unique_pools_for_query(example: dict[str, Any], *, rng: Any) -> list[list[tuple[Any, int]]]:
    pool = example["candidate_pool"]
    scores = example["candidate_scores"]

    neg = [(d, int(s)) for d, s in zip(pool, scores) if int(s) == 0]
    pos1 = [(d, int(s)) for d, s in zip(pool, scores) if int(s) == 1]
    pos2 = [(d, int(s)) for d, s in zip(pool, scores) if int(s) >= 2]
    all_pos = pos2 + pos1
    all_docs = all_pos + neg

    if HARD_NEG_TOP is not None and len(neg) > HARD_NEG_TOP:
        neg = neg[:HARD_NEG_TOP]

    if len(all_docs) == 0:
        return []

    modes = ["only_2", "only_1", "mixed", "any", "any", "any"]
    seen = set()
    unique_selected_docs = []

    tries = 0
    while len(unique_selected_docs) < N_VARIANTS and tries < MAX_TOTAL_TRIES:
        mode = modes[tries % len(modes)]
        tries += 1

        positives = []
        if mode == "only_2":
            k = min(len(pos2), MAX_POSITIVES)
            positives = rng.sample(pos2, k) if k > 0 else []
        elif mode == "only_1":
            k = min(len(pos1), MAX_POSITIVES)
            positives = rng.sample(pos1, k) if k > 0 else []
        elif mode == "mixed":
            k2 = min(len(pos2), MAX_POSITIVES // 2)
            k1 = min(len(pos1), MAX_POSITIVES - k2)
            if k2 > 0:
                positives += rng.sample(pos2, k2)
            if k1 > 0:
                positives += rng.sample(pos1, k1)
        else:
            k = min(len(all_pos), MAX_POSITIVES)
            positives = rng.sample(all_pos, k) if k > 0 else []

        needed = TARGET_SIZE - len(positives)
        if needed < 0:
            positives = positives[:TARGET_SIZE]
            needed = 0

        negatives = []
        if needed > 0:
            kneg = min(len(neg), needed)
            negatives = rng.sample(neg, kneg) if kneg > 0 else []

        if not positives and all_pos and MAX_POSITIVES > 0:
            positives = [rng.choice(all_pos)]

        selected = positives + negatives

        if len(selected) < TARGET_SIZE:
            used_ids = {d for d, _ in selected}
            remaining = [p for p in all_docs if p[0] not in used_ids]
            selected += remaining[: max(0, TARGET_SIZE - len(selected))]

        sig = training_pool_signature([d for d, _ in selected])
        if sig in seen:
            continue

        seen.add(sig)
        rng.shuffle(selected)
        unique_selected_docs.append(selected)

    return unique_selected_docs


prompt_level_doc_ids: set[str] = set()
prompt_level_positive_doc_ids: set[str] = set()
prompt_level_examples = 0
prompt_level_score_counter = Counter()

positive_rows = []
for row_id, row in enumerate(iter_jsonl(HRIA / "reranking_dataset_doris_mae.jsonl")):
    if not any(int(score) > 0 for score in row["candidate_scores"]):
        continue
    positive_rows.append((row_id, row))
    rng = random.Random(training_stable_seed(row_id, first_nonempty(row.get("query_text"))))
    selected_pools = sample_training_unique_pools_for_query(row, rng=rng)
    for selected_docs in selected_pools:
        prompt_level_examples += 1
        for doc_id, score in selected_docs:
            doc_id = str(doc_id)
            score = int(score)
            prompt_level_doc_ids.add(doc_id)
            prompt_level_score_counter[score] += 1
            if score > 0:
                prompt_level_positive_doc_ids.add(doc_id)

prompt_level_docs = [make_doris_record(doc_id, "actual_prompt_candidate") for doc_id in sorted(prompt_level_doc_ids, key=int)]
prompt_level_positive_docs = [make_doris_record(doc_id, "actual_prompt_positive_candidate") for doc_id in sorted(prompt_level_positive_doc_ids, key=int)]

prompt_exposure_stats = {
    "filtered_positive_aspect_rows": len(positive_rows),
    "prompt_training_examples": prompt_level_examples,
    "target_size": TARGET_SIZE,
    "unique_prompt_candidate_docs": len(prompt_level_docs),
    "unique_prompt_positive_candidate_docs": len(prompt_level_positive_docs),
    "score_mentions": dict(sorted(prompt_level_score_counter.items())),
}

print(json.dumps(prompt_exposure_stats, indent=2))


In [ ]:
# -----------------------------
# Prompt-level exact overlap comparisons
# -----------------------------

prompt_level_comparisons = [
    run_exact_comparison("DORIS actual prompt candidates", prompt_level_docs, "CSFCube eval docs", csf_eval_docs),
    run_exact_comparison("DORIS actual prompt positive candidates", prompt_level_positive_docs, "CSFCube eval docs", csf_eval_docs),
    run_exact_comparison("DORIS actual prompt candidates", prompt_level_docs, "CSFCube query docs", [r for r in csf_eval_docs if "query" in r["role"]]),
    run_exact_comparison("DORIS actual prompt candidates", prompt_level_docs, "CSFCube candidate docs", [r for r in csf_eval_docs if "candidate" in r["role"]]),
    run_exact_comparison("DORIS actual prompt candidates", prompt_level_docs, "LitSearch local IDs", lit_docs),
    run_exact_comparison("DORIS actual prompt positive candidates", prompt_level_positive_docs, "LitSearch local IDs", lit_docs),
]

if lit_text_docs:
    prompt_level_comparisons.extend([
        run_exact_comparison("DORIS actual prompt candidates", prompt_level_docs, "LitSearch text docs", lit_text_docs),
        run_exact_comparison("DORIS actual prompt positive candidates", prompt_level_positive_docs, "LitSearch text docs", lit_text_docs),
    ])

prompt_level_summary_rows = [
    {k: v for k, v in comparison.items() if k != "matches"}
    for comparison in prompt_level_comparisons
]

prompt_level_detail_rows = []
for comparison in prompt_level_comparisons:
    for field, rows in comparison["matches"].items():
        for row in rows:
            prompt_level_detail_rows.append(row)

write_csv(OUT_DIR / "prompt_level_overlap_summary.csv", prompt_level_summary_rows)
write_json(OUT_DIR / "prompt_level_overlap_summary.json", prompt_level_summary_rows)
write_csv(OUT_DIR / "prompt_level_overlap_details.csv", prompt_level_detail_rows)
write_json(OUT_DIR / "prompt_level_overlap_details.json", prompt_level_detail_rows)
write_json(OUT_DIR / "prompt_level_exposure_stats.json", prompt_exposure_stats)

maybe_display(prompt_level_summary_rows)
print("Prompt-level exact match detail rows:", len(prompt_level_detail_rows))


## Query-level overlap analysis

This section checks whether the *query texts / information needs* overlap, separately from document-level overlap.

For DORIS-MAE, the training-time query text is the aspect-level `query_text` used to build prompts. The original 100 complex DORIS queries are also included for reference. For LitSearch, the query text is the natural-language literature-search question. For CSFCube, the query is query-by-example, so the notebook represents each query as the query paper title plus facet, and also keeps a fuller title+abstract+facet descriptor.

In [ ]:
# -----------------------------
# Query records
# -----------------------------

def make_query_record(dataset: str, role: str, query_id: str, text: str, **extra: Any) -> dict[str, Any]:
    normalized = normalize_text(text)
    tokens = [t for t in normalized.split() if len(t) > 2]
    return {
        "dataset": dataset,
        "role": role,
        "query_id": str(query_id),
        "query_text": text,
        "query_hash": stable_hash(text),
        "normalized_query": normalized,
        "token_set": set(tokens),
        **extra,
    }

# DORIS original complex queries.
doris_original_queries = []
for i, row in enumerate(load_json(DORIS_DIR / "Query.json")):
    doris_original_queries.append(
        make_query_record(
            "DORIS-MAE",
            "original_complex_query",
            f"doris-original-{i}",
            first_nonempty(row.get("query_text")),
            query_type=first_nonempty(row.get("query_type")),
        )
    )

# DORIS aspect-level training queries; these are what the finetuning prompts actually use.
doris_aspect_query_by_text = {}
for i, row in enumerate(iter_jsonl(HRIA / "reranking_dataset_doris_mae.jsonl")):
    text = first_nonempty(row.get("query_text"))
    key = normalize_text(text)
    if key not in doris_aspect_query_by_text:
        doris_aspect_query_by_text[key] = make_query_record(
            "DORIS-MAE",
            "aspect_level_training_query",
            f"doris-aspect-{len(doris_aspect_query_by_text)}",
            text,
            aspect_id=first_nonempty(row.get("aspect_id")),
            query_type=first_nonempty(row.get("original_query_type")),
        )

doris_aspect_queries = list(doris_aspect_query_by_text.values())

# LitSearch query strings from local retrieval/reranking outputs.
lit_query_by_text = {}
for path in lit_result_files:
    if "metrics" in path.name:
        continue
    for row in iter_jsonl(path):
        text = first_nonempty(row.get("query"))
        key = normalize_text(text)
        if key and key not in lit_query_by_text:
            lit_query_by_text[key] = make_query_record(
                "LitSearch",
                "literature_search_query",
                f"litsearch-{len(lit_query_by_text)}",
                text,
                query_set=first_nonempty(row.get("query_set")),
                specificity=first_nonempty(row.get("specificity")),
            )

lit_queries = list(lit_query_by_text.values())

# CSFCube query-by-example descriptors. The actual query is a paper plus facet, not a natural-language question.
facet_names = {
    "background": "background/objective",
    "method": "method",
    "result": "result",
}
csf_query_short = []
csf_query_full = []
seen_csf_query_facet = set()
for pid in sorted(csf_query_ids, key=int):
    doc = csf_text_by_pid.get(pid, {})
    title = first_nonempty(doc.get("title"))
    abstract = first_nonempty(doc.get("abstract"))
    for facet in sorted(csf_facets.get(pid, [])):
        key = (pid, facet)
        if key in seen_csf_query_facet:
            continue
        seen_csf_query_facet.add(key)
        facet_text = facet_names.get(facet, facet)
        csf_query_short.append(
            make_query_record(
                "CSFCube",
                "query_paper_title_plus_facet",
                f"csfcube-{pid}-{facet}-short",
                f"Facet: {facet_text}. Query paper title: {title}",
                paper_id=pid,
                facet=facet,
            )
        )
        csf_query_full.append(
            make_query_record(
                "CSFCube",
                "query_paper_title_abstract_plus_facet",
                f"csfcube-{pid}-{facet}-full",
                f"Facet: {facet_text}. Query paper title: {title}. Query paper abstract: {abstract}",
                paper_id=pid,
                facet=facet,
            )
        )

print("DORIS original queries:", len(doris_original_queries))
print("DORIS unique aspect-level training queries:", len(doris_aspect_queries))
print("LitSearch unique query strings:", len(lit_queries))
print("CSFCube query title+facet descriptors:", len(csf_query_short))
print("CSFCube full query descriptors:", len(csf_query_full))


In [ ]:
# -----------------------------
# Exact and near query matching
# -----------------------------

def query_exact_matches(left_name: str, left_records: list[dict[str, Any]], right_name: str, right_records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    left_index = index_by_field(left_records, "query_hash")
    right_index = index_by_field(right_records, "query_hash")
    rows = []
    for value in sorted(set(left_index) & set(right_index)):
        for left in left_index[value]:
            for right in right_index[value]:
                rows.append({
                    "match_type": "exact_query_hash",
                    "score": 1.0,
                    "left_set": left_name,
                    "left_query_id": left["query_id"],
                    "left_role": left["role"],
                    "left_query_text": left["query_text"][:500],
                    "right_set": right_name,
                    "right_query_id": right["query_id"],
                    "right_role": right["role"],
                    "right_query_text": right["query_text"][:500],
                })
    return rows


def query_similarity(left: dict[str, Any], right: dict[str, Any]) -> tuple[float, float]:
    seq_score = SequenceMatcher(None, left["normalized_query"], right["normalized_query"]).ratio()
    left_tokens = left["token_set"]
    right_tokens = right["token_set"]
    if not left_tokens or not right_tokens:
        return seq_score, 0.0
    jaccard = len(left_tokens & right_tokens) / len(left_tokens | right_tokens)
    return seq_score, jaccard


def query_near_matches(
    left_name: str,
    left_records: list[dict[str, Any]],
    right_name: str,
    right_records: list[dict[str, Any]],
    *,
    seq_threshold: float = 0.82,
    jaccard_threshold: float = 0.55,
) -> list[dict[str, Any]]:
    rows = []
    for left in left_records:
        for right in right_records:
            seq_score, jaccard = query_similarity(left, right)
            if seq_score >= seq_threshold or jaccard >= jaccard_threshold:
                rows.append({
                    "match_type": "near_query_text",
                    "sequence_score": round(seq_score, 4),
                    "jaccard_score": round(jaccard, 4),
                    "left_set": left_name,
                    "left_query_id": left["query_id"],
                    "left_role": left["role"],
                    "left_query_text": left["query_text"][:500],
                    "right_set": right_name,
                    "right_query_id": right["query_id"],
                    "right_role": right["role"],
                    "right_query_text": right["query_text"][:500],
                })
    return sorted(rows, key=lambda row: (row["sequence_score"], row["jaccard_score"]), reverse=True)

query_comparison_specs = [
    ("DORIS original queries", doris_original_queries, "LitSearch queries", lit_queries),
    ("DORIS aspect training queries", doris_aspect_queries, "LitSearch queries", lit_queries),
    ("DORIS original queries", doris_original_queries, "CSFCube title+facet queries", csf_query_short),
    ("DORIS aspect training queries", doris_aspect_queries, "CSFCube title+facet queries", csf_query_short),
]

query_exact_rows = []
query_near_rows = []
query_summary_rows = []

for left_name, left_records, right_name, right_records in query_comparison_specs:
    exact = query_exact_matches(left_name, left_records, right_name, right_records)
    near = query_near_matches(left_name, left_records, right_name, right_records)
    query_exact_rows.extend(exact)
    query_near_rows.extend(near)
    query_summary_rows.append({
        "left_set": left_name,
        "right_set": right_name,
        "left_queries": len(left_records),
        "right_queries": len(right_records),
        "exact_query_hash_pairs": len(exact),
        "exact_left_queries": len({row["left_query_id"] for row in exact}),
        "exact_right_queries": len({row["right_query_id"] for row in exact}),
        "near_query_pairs": len(near),
        "near_left_queries": len({row["left_query_id"] for row in near}),
        "near_right_queries": len({row["right_query_id"] for row in near}),
    })

write_csv(OUT_DIR / "query_overlap_summary.csv", query_summary_rows)
write_json(OUT_DIR / "query_overlap_summary.json", query_summary_rows)
write_csv(OUT_DIR / "query_exact_overlap_details.csv", query_exact_rows)
write_json(OUT_DIR / "query_exact_overlap_details.json", query_exact_rows)
write_csv(OUT_DIR / "query_near_overlap_candidates.csv", query_near_rows)
write_json(OUT_DIR / "query_near_overlap_candidates.json", query_near_rows)

maybe_display(query_summary_rows)
print("Exact query matches:", len(query_exact_rows))
print("Near query candidates:", len(query_near_rows))
if query_near_rows:
    maybe_display(query_near_rows, limit=20)


## Dataset-level report

This cell writes a compact report that can be adapted into the thesis.

In [ ]:
report = {
    "doris": {
        "aspect_level_rows": doris_rows,
        "corpus_docs": len(doris_corpus_docs),
        "train_candidate_docs": len(doris_train_docs),
        "positive_train_candidate_docs": len(doris_positive_docs),
    },
    "csfcube": {
        "text_docs": len(csf_text_docs),
        "eval_docs": len(csf_eval_docs),
        "query_docs": len(csf_query_ids),
        "candidate_docs": len(csf_candidate_ids),
    },
    "litsearch": {
        "local_result_files": len(lit_result_files),
        "local_ids": len(lit_ids),
        "gold_relevant_ids": len(lit_gold_ids),
        "text_source": lit_text_source,
        "text_docs": len(lit_text_docs),
    },
    "exact_overlap_summary": summary_rows,
    "exact_overlap_details_count": len(detail_rows),
    "near_duplicate_title_candidates_count": len(near_rows),
    "prompt_level_exposure": prompt_exposure_stats,
    "prompt_level_overlap_summary": prompt_level_summary_rows,
    "prompt_level_overlap_details_count": len(prompt_level_detail_rows),
    "query_overlap_summary": query_summary_rows,
    "query_exact_overlap_details_count": len(query_exact_rows),
    "query_near_overlap_candidates_count": len(query_near_rows),
}

write_json(OUT_DIR / "leakage_report.json", report)

print(json.dumps(report, indent=2, ensure_ascii=False)[:5000])
print("\nWrote reports to:", OUT_DIR)


## Thesis wording template

Use this after inspecting the generated `exact_overlap_summary.csv` and `exact_overlap_details.csv` files.

> We performed a document-level leakage analysis between the DORIS-MAE-derived fine-tuning data and the LitSearch / CSFCube evaluation material. The DORIS-MAE training exposure set was defined conservatively as the union of all documents appearing in the DORIS-MAE aspect-level candidate pools, rather than only the sampled prompt variants. We compared this set against CSFCube query and candidate documents, and against all locally available LitSearch retrieved, reranked, and gold-relevant document IDs. Matching was performed using available metadata identifiers, normalized arXiv/DOI identifiers, normalized-title hashes, normalized title+abstract hashes, and a secondary near-duplicate title screen. Raw integer IDs were not interpreted as conclusive across datasets because the datasets use different identifier namespaces.
>
> No exact title or title+abstract overlap was found between the DORIS-MAE training-exposed documents and the CSFCube evaluation documents [update if generated report differs]. LitSearch text-level matching was performed when the LitSearch corpus was locally available; otherwise the local LitSearch analysis was limited to the IDs present in retrieval/reranking outputs and should be interpreted as an identifier-level sanity check rather than a full text-level duplicate search [update according to report].
